# PS- 3D: Easy LSTM Autoencoder Version

This is the LSTM comparison version. The existing PS- 3D.ipynb remains the simpler dense-autoencoder version.


In [1]:
# Import all required packages first
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report
from tensorflow.keras import models, layers


In [2]:
# 1. Load data, separate x and y, then keep healthy rows for training.
data = pd.read_csv("NASA TurboFan Engine.csv")
x = data.drop(["unit", "cycle", "RUL", "Fault_Label"], axis=1)
y = data["Fault_Label"]
x_train, x_test, y_train, y_test = train_test_split(
    x, y, test_size=0.2, random_state=42, stratify=y
)
normal = x_train[y_train == 0]
scaler = StandardScaler()
normal = scaler.fit_transform(normal)
x_test = scaler.transform(x_test)
x_train, x_val = train_test_split(normal, test_size=0.1, random_state=42)

# LSTM needs 3D input: samples, time steps, features.
# Here each row's sensor values are treated as a short sequence.
x_train = x_train.reshape(-1, x_train.shape[1], 1)
x_val = x_val.reshape(-1, x_val.shape[1], 1)
x_test = x_test.reshape(-1, x_test.shape[1], 1)


In [3]:
# 2. Create the LSTM encoder and decoder.
encoder = models.Sequential([
    layers.Input(shape=(x_train.shape[1], 1)),
    layers.LSTM(16)
])
decoder = models.Sequential([
    layers.Input(shape=(16,)),
    layers.RepeatVector(x_train.shape[1]),
    layers.LSTM(16, return_sequences=True),
    layers.TimeDistributed(layers.Dense(1))
])
model = models.Sequential([encoder, decoder])
model.compile(optimizer="adam", loss="mse", metrics=["mae"])


In [ ]:
# 3. Train the LSTM autoencoder only on healthy data.
model.fit(x_train, x_train, validation_data=(x_val, x_val),
          epochs=20, batch_size=512, verbose=2)


Epoch 1/20
25/25 - 23s - 909ms/step - loss: 0.6972 - mae: 0.5540 - val_loss: 0.6695 - val_mae: 0.5625
Epoch 2/20
25/25 - 2s - 97ms/step - loss: 0.6786 - mae: 0.5720 - val_loss: 0.6609 - val_mae: 0.5658
Epoch 3/20
25/25 - 2s - 93ms/step - loss: 0.6654 - mae: 0.5715 - val_loss: 0.6436 - val_mae: 0.5712
Epoch 4/20
25/25 - 2s - 99ms/step - loss: 0.6475 - mae: 0.5718 - val_loss: 0.6240 - val_mae: 0.5668
Epoch 5/20
25/25 - 2s - 94ms/step - loss: 0.6303 - mae: 0.5661 - val_loss: 0.6105 - val_mae: 0.5597
Epoch 6/20
25/25 - 2s - 95ms/step - loss: 0.6204 - mae: 0.5589 - val_loss: 0.6043 - val_mae: 0.5547
Epoch 7/20
25/25 - 2s - 93ms/step - loss: 0.6149 - mae: 0.5543 - val_loss: 0.6010 - val_mae: 0.5512
Epoch 8/20
25/25 - 3s - 104ms/step - loss: 0.6118 - mae: 0.5512 - val_loss: 0.6010 - val_mae: 0.5506
Epoch 9/20
25/25 - 2s - 97ms/step - loss: 0.6105 - mae: 0.5499 - val_loss: 0.5942 - val_mae: 0.5442
Epoch 10/20
25/25 - 2s - 93ms/step - loss: 0.6074 - mae: 0.5470 - val_loss: 0.5923 - val_mae: 0.5

In [ ]:
# 4. Calculate reconstruction error and detect faulty rows.
threshold = np.percentile(np.mean((x_val - model.predict(x_val, verbose=0)) ** 2, axis=(1, 2)), 99)
error = np.mean((x_test - model.predict(x_test, verbose=0)) ** 2, axis=(1, 2))
prediction = (error > threshold).astype(int)
print("Threshold:", threshold)
print(classification_report(y_test, prediction, target_names=["Healthy", "Fault"], zero_division=0))
print("Number of anomalies detected:", prediction.sum())
